# Comprehensive SFT Guide

This notebook provides a comprehensive guide to Supervised Fine-Tuning (SFT) using LLaMA-Factory, covering:

1. **Dataset Types**: Instruction-response, conversation, tool calling, multimodal
2. **Training Methods**: Full fine-tuning, LoRA, QLoRA
3. **Advanced Features**: Multi-turn conversations, custom datasets
4. **Evaluation**: Performance benchmarking and quality assessment

## Table of Contents

- [Setup and Installation](#setup-and-installation)
- [Dataset Preparation](#dataset-preparation)
- [Training Configuration](#training-configuration)
- [Model Training](#model-training)
- [Evaluation and Benchmarking](#evaluation-and-benchmarking)
- [Advanced Features](#advanced-features)
- [Best Practices](#best-practices)


## Setup and Installation

First, let's install the required dependencies and set up the environment.


In [ ]:
# Install LLaMA-Factory and dependencies
%pip install -r requirements.txt
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Clone LLaMA-Factory (if not already cloned)
# !git clone https://github.com/hiyouga/LLaMA-Factory.git
# !cd LLaMA-Factory


In [ ]:
# Import required libraries
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from llamafactory import ChatModel
import json
import os

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")


## Dataset Preparation

Let's prepare different types of datasets for SFT training.


In [ ]:
# 1. Instruction-Response Dataset (Alpaca-style)
instruction_response_data = [
    {
        "instruction": "Explain the concept of machine learning in simple terms.",
        "input": "",
        "output": "Machine learning is a type of artificial intelligence that allows computers to learn and improve from experience without being explicitly programmed. It works by finding patterns in data and using those patterns to make predictions or decisions. Think of it like teaching a child - instead of giving step-by-step instructions, you show them many examples and they learn to recognize patterns and make decisions based on what they've seen."
    },
    {
        "instruction": "Write a Python function to calculate the factorial of a number.",
        "input": "n = 5",
        "output": "def factorial(n):\n    if n == 0 or n == 1:\n        return 1\n    else:\n        return n * factorial(n-1)\n\nresult = factorial(5)\nprint(result)  # Output: 120"
    }
]

# Save instruction-response dataset
with open('data/instruction_response_demo.json', 'w') as f:
    json.dump(instruction_response_data, f, indent=2)


In [ ]:
# 2. Conversation Dataset (ShareGPT-style)
conversation_data = [
    {
        "conversations": [
            {"from": "human", "value": "Hello! Can you help me understand quantum computing?"},
            {"from": "gpt", "value": "Hello! I'd be happy to help you understand quantum computing. Quantum computing is a revolutionary approach to computation that leverages the principles of quantum mechanics. Unlike classical computers that use bits (0s and 1s), quantum computers use quantum bits or 'qubits' that can exist in multiple states simultaneously thanks to a property called superposition.\n\nThis allows quantum computers to process vast amounts of information in parallel, making them potentially much faster for certain types of problems like:\n\n1. **Cryptography** - Breaking current encryption methods\n2. **Drug discovery** - Simulating molecular interactions\n3. **Optimization** - Finding optimal solutions to complex problems\n4. **Machine learning** - Training more sophisticated AI models\n\nWould you like me to explain any of these applications in more detail, or do you have questions about specific quantum concepts like superposition, entanglement, or quantum gates?"}
        ]
    }
]

# Save conversation dataset
with open('data/conversation_demo.json', 'w') as f:
    json.dump(conversation_data, f, indent=2)


In [ ]:
# 3. Tool Calling Dataset
tool_calling_data = [
    {
        "conversations": [
            {"from": "human", "value": "What's the weather like in New York today?"},
            {"from": "gpt", "value": "First, the user is asking about the weather in New York. I need to use the get_weather tool to fetch current weather information."}
        ],
        "tools": [
            {
                "name": "get_weather",
                "description": "Get current weather information for a city",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "city": {"type": "string", "description": "City name"}
                    },
                    "required": ["city"]
                }
            }
        ]
    }
]

# Save tool calling dataset
with open('data/tool_calling_demo.json', 'w') as f:
    json.dump(tool_calling_data, f, indent=2)
